<div align="center">

# **LLM Visual Explorer V1**
## (PL, Aug26)

![LlmExpl — Visual Explorer](../images/LlmExpl_logo.png)

## Explorer l'espace d'un modèle de langage à travers un "trou de serrure"

</div>

Un mot n'est pas un point isolé. Il possède une position dans un
espace sémantique de plusieurs centaines, voire milliers de dimensions.

---

### Cellules du Notebook

0. **Initialisation**

1. **Scénario**  
   Choix des concepts à explorer.

2. **Embeddings**  
   Chargement du modèle et création des vecteurs sémantiques.

3. **Projection 3D + DataFrame**  
   Réduction de l'espace multidimensionnel et préparation des données.

4. **Visualisation 2D — "Carte sémantique"**
   Exploration des concepts en 2D

5. **Visualisation 3D — "Planétarium sémantique"**
   Exploration des concepts en 3D

6. **Similarités — Mesure des proximités sémantiques**  
   Classement des paires de concepts, du plus au moins similaire.

7. **Rapport final**  
   Synthèse et conservation des résultats de l'exploration.

In [1]:
#=================================================
# 0 - Initialization
#=================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from explorer.notebook_setup import *

Project root: c:\Users\pefsy\Projects\LLM-Visual-Explorer


In [10]:
#=================================================
#  1 - Chargement du Scenario
#=================================================
'''
# Choix du scénario super-scenario for introduction
   animals, countries, geek_tools, multilingual,
   plants, professions, scientists, verbs
'''
SCENARIO_NAME = "super-scenario" 

scenario = load_scenario(SCENARIO_NAME)

display_scenario(scenario)

concepts = [
    obj["name"]
    for obj in scenario["objects"]
]

categories = [
    obj["category"]
    for obj in scenario["objects"]
]

objects = scenario["objects"]

icons = {
    obj["name"]: obj.get("emoji", "")
    for obj in objects
}


SCÉNARIO : Super scénario

Exploration d'un espace sémantique mélangeant objets, êtres vivants, métiers et concepts abstraits.

📋 Concepts analysés :

   🌳  Arbre
   🐶  Chien
   🐦  Oiseau
   🌸  Fleur
   🐬  Dauphin
   🏠  Maison
   🚗  Voiture
   💻  Ordinateur
   📖  Livre
   🔨  Marteau
   👷  Ingénieur
   🔬  Chercheur
   👨‍🏫  Professeur
   ⚖️  Avocat
   ✍️  Écrire
   🗣️  Parler
   ❤️  Amour
   🧠  Apprendre
   🕊️  Liberté

Nombre d'objets : 19


In [11]:
# ==========================================================
# 2 Chargement du modèle (embeddings)
# ==========================================================

# Chargement du modèle
model = SentenceTransformer(MODEL_NAME) # Choix parmi 3 modèles dans config.py

# Calcul des embeddings
embeddings = compute_embeddings(concepts)


# Affichage des informations
display_model(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MODÈLE D'EMBEDDINGS

Modèle :
Nom Générique: MPNet

Dimension des embeddings :
  768



In [ ]:
#================================================
# 3 - Projection 3D & DataFrame
#================================================

xyz, pca = compute_pca(
    embeddings
)

display_projection(
    pca
)

df = create_dataframe(
    concepts,
    categories,
    xyz,
)

PROJECTION 3D



TypeError: unsupported format string passed to PCA.__format__

In [ ]:
#================================================
# 4 Semantic map 2D
#================================================
# Cette carte est une projection des concepts dans un espace mathématique.
# Deux points proches représentent des concepts dont les représentations 
# internes sont proches.

fig_map = plot_map(
    df=df,
    title="2D semantic Map",
    icons=icons,
    show_labels=True,
    show_icons=False,
)

fig_map.show()

In [ ]:
#==============================================
# 5 Semantic Galaxy 3D
#================================================


icons = {
    obj["name"]: obj.get("emoji", "")
    for obj in scenario["objects"]
}


fig = plot_scene(
    df=df,
    title="3D Semantic Planetarium",
    icons=icons,
    show_labels=True,
    show_icons=False,
)


fig.show()

In [ ]:
#================================================
# 6   Classement des similarités
#================================================

from explorer.exports import save_similarities_csv

similarities = compute_similarity(embeddings)

similarity_pairs = rank_similarity_pairs(
    concepts,
    similarities
)

display_similarity_ranking(
    similarity_pairs,
    df,
    icons,
    top_n=10,
    bottom_n=10,
)

save_similarities_csv(
    similarity_pairs,
    scenario_name=scenario["title"],
    model_alias=MODEL_ALIAS,
    model_name=MODEL_NAME,
)



🔗 SEMANTIC SIMILARITY vs 3D PROJECTION DISTANCE

Seuils indicatifs:
🟢 60–100 %   Similarité élevée
🟡 30–59 %    Similarité moyenne
🔴 0–29 %     Similarité faible

NB: L'échelle des similarités dépend du modèle. Le classement est plus significatif que la valeur absolue.

🟢 Les 10 voisins sémantiques
--------------------------------------------------------------------------------
     1. 🩺 Médecin      ↔ 💉 Infirmier    ████████████         Sim: 64%  Dist: 1.2
     2. 🥖 Boulanger    ↔ 🪚 Menuisier    ████████████         Sim: 60%  Dist: 1.1
     3. 🔬 Chercheur    ↔ 🥖 Boulanger    ██████████           Sim: 55%  Dist: 1.0
     4. 🔬 Chercheur    ↔ 🪚 Menuisier    ██████████           Sim: 54%  Dist: 1.9
     5. 🩺 Médecin      ↔ 🪚 Menuisier    ██████████           Sim: 54%  Dist: 1.0
     6. 👷 Ingénieur    ↔ 🪚 Menuisier    ██████████           Sim: 53%  Dist: 0.7
     7. 🪚 Menuisier    ↔ ✈️ Pilote       ██████████           Sim: 52%  Dist: 0.8
     8. 👷 Ingénieur    ↔ ✈️ Pilote       █████████

In [ ]:
#======================================================================
# 7 Display exploration report
#======================================================================


display_report(
    scenario_name=SCENARIO_NAME,
    model_name=MODEL_NAME,
    concepts=concepts,
    embedding_dimension=embeddings.shape[1],
    pca=pca,
    similarity_pairs=similarity_pairs,
)


🌌 LLM VISUAL EXPLORER REPORT

Scenario              : professions
Model alias           : MPNet
Technical Model       : sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Number of concepts    : 10
Embedding dimension   : 768

----------------------------------------------------------------------
📐 PROJECTION 3D (PCA)
----------------------------------------------------------------------

768 dimensions → 3D préserve 55.4% de la structure sémantique


----------------------------------------------------------------------
🔗 OBSERVATIONS SEMANTIQUES
----------------------------------------------------------------------

🥇 Paire la plus proche : Médecin ↔ Infirmier (64%)
🔻 Paire la plus éloignée: Avocat ↔ Boulanger (13%)

----------------------------------------------------------------------
💾 DATA EXPORT
----------------------------------------------------------------------

The numerical data used during this exploration can be saved:

  • concepts.csv      → concepts and cate